This notebook first reviews the feature layers and tables in a GIS portal,
determining for each item if it has its "exportable" capability turned ON in its settings.
Then, for those that have exportable=False, it adds that capability.


In [ ]:
import os
import sys


sys.path.insert(0, r'C:\Users\cpeak\repos\data-portal-tools')
from PortalConnector import PortalConnector
from arcgis.features import FeatureLayerCollection

enterprise_client_id = os.getenv("ENTERPRISE_PORTAL_CLIENT_ID")
agol_user = os.getenv('AGOL_ADMIN_USERNAME')
agol_pw = os.getenv('AGOL_ADMIN_PW')
# with open(r'Config\\auth.yml') as file:
# 	auth = yaml.load(file, Loader=yaml.FullLoader)
portal_conn = PortalConnector(
	portal_username=agol_user,
	portal_pw=agol_pw
 )


gis = portal_conn.gis
print('Connected as:', gis.users.me.username)

Connected as: gisadmin_PSRC


In [5]:
grp = gis.groups.search('title:"PSRC Data Portal Content"')[0]
print(grp)

<Group title:"PSRC Data Portal Content" owner:gisadmin_PSRC>


In [6]:
seen = {}
for item_type in ['Feature Layer', 'Table']:
    #items = gis.content.search(query=f'group:{grp.id}', item_type=item_type, max_items=10000)
    items = gis.content.search(query=f'group:{grp.id}', item_type=item_type, max_items=10000)
    #items = grp.content()
    print(f'{item_type}: {len(items)} items found')
    for item in items:
        seen[item.id] = item
        print(f"item {item.title} type: {item.type}")

print(f'Total unique items: {len(seen)}')


c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

Feature Layer: 93 items found
item Transit_Routes_2018_RTP type: Feature Service
item Income_Restricted_Housing_Database_2024_by_Jurisdiction type: Feature Service
item Census_Blocks_2010 type: Feature Service
item Income_Restricted_Housing_Database_2020_by_County type: Feature Service
item Census_Tracts_2000 type: Feature Service
item Income_Restricted_Housing_Database_2024_by_County type: Feature Service
item Macroeconomic_Forecast_2018 type: Feature Service
item LUV-it_TAZ_Summaries_2023 type: Feature Service
item Equity_Focus_Areas_2019 type: Feature Service
item Income_Restricted_Housing_Database_2021_by_County type: Feature Service
item LUV-it_Control_Geogs_and_HCT_Summaries_2023 type: Feature Service
item Equity_Focus_Areas type: Feature Service
item Census_Blocks_2000 type: Feature Service
item Regional_Parking_Inventory type: Feature Service
item County_Boundaries type: Feature Service
item Census_PUMAS_2020 type: Feature Service
item control_hct_2018_land_use_forecast_dashboa

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

Table: 0 items found
Total unique items: 93


In [7]:
rows = []
errors = []
for item in seen.values():
    try:
        print(f'reviewing {item.title}...', end=' ')
        flc = FeatureLayerCollection.fromitem(item)
        caps = flc.properties.get('capabilities', '') or ''
        extract_enabled = 'Extract' in [c.strip() for c in caps.split(',')]
        rows.append((item.title, item.type, item.owner, caps, extract_enabled, item.id))
        print(extract_enabled)
    except Exception as e:
        errors.append((item.title, item.type, str(e)[:150]))

disabled = sorted([r for r in rows if not r[4]], key=lambda r: r[0].lower())
enabled = [r for r in rows if r[4]]

print()
print(f'Extract ENABLED: {len(enabled)}')
print(f'Extract DISABLED: {len(disabled)}')
print()
print('=== Extract DISABLED ===')
for title, typ, owner, caps, _, iid in disabled:
    print(f'{title} | {typ} | owner={owner} | caps={caps} | id={iid}')

if errors:
    print()
    print(f'=== Errors ({len(errors)}) ===')
    for e in errors:
        print(e)


reviewing Transit_Routes_2018_RTP... 

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


True
reviewing Income_Restricted_Housing_Database_2024_by_Jurisdiction... 

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


True
reviewing Census_Blocks_2010... 

...

True
reviewing Census_PUMAS_2000... 

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


True
reviewing LUV-it_County_Summaries_2023... 

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


True

Extract ENABLED: 93
Extract DISABLED: 0

=== Extract DISABLED ===


In [ ]:
# Enable the "Extract" (Export data) capability on every item in `disabled`.
# DRY_RUN = True only reports what would change; set it to False to apply.
DRY_RUN = True

updated, skipped, failed = [], [], []
for title, typ, owner, caps, _, iid in disabled:
    item = seen[iid]
    try:
        flc = FeatureLayerCollection.fromitem(item)
        current = flc.properties.get('capabilities', '') or ''
        cap_list = [c.strip() for c in current.split(',') if c.strip()]
        if 'Extract' in cap_list:
            skipped.append(title)
            continue
        new_caps = ','.join(cap_list + ['Extract'])
        if DRY_RUN:
            print(f'[dry run] {title}: {current} -> {new_caps}')
        else:
            flc.manager.update_definition({'capabilities': new_caps})
            print(f'updated {title}: {new_caps}')
        updated.append(title)
    except Exception as e:
        failed.append((title, str(e)[:150]))

print()
print(f'{"would update" if DRY_RUN else "updated"}: {len(updated)}')
print(f'already had Extract: {len(skipped)}')
print(f'failed: {len(failed)}')
for title, msg in failed:
    print(f'  {title}: {msg}')


updated Annual Housing Development Estimates by Census Tract: Query,Extract
updated Annual Housing Development Estimates by Jurisdiction: Create,Delete,Query,Update,Editing,Extract
updated Annual_Housing_Development_Estimates_by_County: Create,Delete,Query,Update,Editing,Extract
updated cities_2018_land_use_forecast_dashboard: Query,Extract
updated Cities_points5: Query,Extract
updated Cities_points6: Query,Extract
updated Cities_points7: Query,Extract
updated City_Boundaries: Query,Extract
updated Commute_Trip_Reduction_Worksites: Query,Extract
updated Covered_Employment_by_Census_Tract_-_Major_Sector: Query,Extract
updated Covered_Employment_by_City_-_Major_Sector: Query,Extract
updated Covered_Employment_by_City_-_NAICS: Query,Extract
updated Displacement_Risk_Data: Query,Extract
updated Equity_Focus_Areas: Query,Extract
updated Equity_Focus_Areas_2019: Query,Extract
updated high_capacity_transit: Query,Extract
updated Income_Restricted_Housing_Database_2021_by_County: Create,Delete